# Progetto 2 – Timed GNN vs LLM per i puzzle "matto in n"

Domande di ricerca:
1. **Per quali n** la timed GNN guida meglio di un LLM?
2. **L'informazione temporale** migliora le prestazioni della GNN?

Tutto il progetto è in questo notebook:

| Sezione | Contenuto |
|---|---|
| 0 | Setup e configurazione |
| 1 | Dati: puzzle Lichess di matto, n = 1…10 |
| 2 | Dagli scacchi ai grafi di TimeGNN |
| 3 | Valutazione: far risolvere i puzzle |
| 4 | Training delle varianti GNN (M1, M2, M3, M4, M4c) |
| 5 | Baseline (random, greedy) |
| 6 | LLM via Ollama |
| 7 | Risultati e test statistici |
| 8 | Foglio per la valutazione umana |

Con `SMOKE = True` tutto gira in pochi minuti su pochi dati, per controllare che funzioni. Poi si mette `False`.

## 0. Setup

Da terminale, una volta sola:
```bash
python3.12 -m venv .venv && source .venv/bin/activate
pip install torch torch_geometric "pandas<3" python-chess zstandard pyarrow scipy matplotlib seaborn requests ipykernel
pip install ./TimeGNN-main          # la libreria, non modificata
brew install stockfish ollama
ollama serve                        # in un terminale a parte
ollama pull qwen3:8b
```

- **pandas deve essere < 3.** Con pandas 3 la libreria azzera i tempi: crea i timestamp in microsecondi e poi li tratta come nanosecondi.
- **Non installare l'extra `[pyg]` di TimeGNN.** torch-scatter e torch-sparse su Mac vanno compilati e non servono.

In [ ]:
import os, io, re, json, time, glob, copy, random, shutil, warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests, zstandard
import chess, chess.engine
import torch
from torch.utils.data import DataLoader
from torch_geometric.data import Batch
from torch_geometric.nn import MessagePassing
from scipy.stats import binomtest, wilcoxon

from timegnn import EventLogTransformer
from timegnn.data.pyg import CustomDataset, custom_collate_fn
from timegnn.models.training import train_epoch, evaluate_epoch
from timegnn.train.early_stopping import EarlyStopping
from timegnn.models.gat_basic import DualGATModel
from timegnn.models.gat_time_decay import DualGATTimeAwareModel, TimeAwareGATConv

# Compatibilità con PyG >= 2.5 (nessun file della libreria viene modificato).
# PyG genera il metodo `propagate` per ogni classe di layer. TimeAwareGATConv eredita da
# GATConv: se viene creato un GATConv (modelli basic) prima di un TimeAwareGATConv,
# quest'ultimo eredita il propagate di GATConv e va in errore
# ("unexpected keyword argument 'edge_attr'"). Gli assegniamo quello generico di PyG,
# cioè quello che usa già quando viene creato per primo.
TimeAwareGATConv.propagate = MessagePassing.propagate

assert pd.__version__ < "3", "serve pandas < 3"
print("pandas", pd.__version__, "| torch", torch.__version__, "| MPS:", torch.backends.mps.is_available())

In [ ]:
SMOKE = True          # True = prova veloce su pochi dati; False = esperimento completo

SEED = 42             # seed dei DATI (campionamento, split, tempi simulati)
SEEDS = [0] if SMOKE else [0, 1, 2]                 # seed del training (3 ripetizioni)
PER_N = 200 if SMOKE else 12000                     # puzzle per n = 1..5 (n = 5 ne ha meno)
TEST_PER_N = 15 if SMOKE else 300                   # puzzle di test per n = 1..5
LONG_PER_N = 5 if SMOKE else 100                    # puzzle di test per n = 6..10
LLM_PER_N = 2 if SMOKE else 100                     # puzzle di test n = 1..5 per l'LLM (+ tutti i lunghi)

EPOCHS = 2 if SMOKE else 40
MIN_EPOCHS = 0 if SMOKE else 10   # il time-decay resta qualche epoca fermo prima di imparare
PATIENCE = 6
LR, BATCH = 1e-3, 128
LAMBDA = 1.0          # decadimento e^(-λt): il default della libreria (0.01) è quasi nullo

DEVICE = "mps" if torch.backends.mps.is_available() else "cpu"
STOCKFISH = shutil.which("stockfish") or "/usr/local/bin/stockfish"   # oppure il percorso di `which stockfish`
LLM_MODEL = "qwen3:8b"
LLM_THINK = False     # ragionamento interno spento (acceso: risposte di migliaia di token)

OUT = "results_smoke" if SMOKE else "results"
os.makedirs("data", exist_ok=True)
os.makedirs(OUT, exist_ok=True)
print("modalità:", "SMOKE" if SMOKE else "COMPLETA", "| device:", DEVICE, "| risultati in", OUT)

## 1. Dati: puzzle Lichess di matto

Il database Lichess (circa 300 MB) contiene per ogni puzzle `FEN`, `Moves` (UCI), `Rating`, `Themes` e altro.

- La **prima mossa** di `Moves` è dell'avversario; poi si alternano risolutore e difensore. Un matto in n ha 2n mosse.
- Il tema `mateIn1` … `mateIn4` indica esattamente n. Il tema `mateIn5` invece vuol dire "**5 o più**": da lì ricaviamo anche i matti lunghi (n = 6…10).

Il file viene letto a blocchi senza decomprimerlo su disco, e il risultato filtrato viene salvato in `data/mates.parquet`.

In [ ]:
RAW = "data/lichess_db_puzzle.csv.zst"
if not os.path.exists(RAW):
    with requests.get("https://database.lichess.org/lichess_db_puzzle.csv.zst", stream=True) as r:
        with open(RAW, "wb") as f:
            for chunk in r.iter_content(1 << 20):
                f.write(chunk)

if os.path.exists("data/mates.parquet"):
    mates = pd.read_parquet("data/mates.parquet")
else:
    cols = ["PuzzleId", "FEN", "Moves", "Rating", "RatingDeviation", "Popularity", "Themes", "GameUrl"]
    parts = []
    with open(RAW, "rb") as fh:
        text = io.TextIOWrapper(zstandard.ZstdDecompressor().stream_reader(fh), encoding="utf-8")
        for chunk in pd.read_csv(text, usecols=cols, chunksize=500_000):
            k = pd.to_numeric(chunk["Themes"].str.extract(r"\bmateIn(\d)\b", expand=False))
            n = chunk["Moves"].str.split().str.len() // 2
            # mateIn1..4 = esattamente n; mateIn5 = 5 o più (teniamo fino a 10)
            keep = (n == k) | ((k == 5) & n.between(6, 10))
            parts.append(chunk[keep].assign(n=n[keep]))
    mates = pd.concat(parts, ignore_index=True)
    mates.to_parquet("data/mates.parquet")

mates["game"] = mates["GameUrl"].str.extract(r"lichess\.org/(\w{8})", expand=False)
mates = mates.drop_duplicates("FEN").drop_duplicates("game")   # niente posizioni o partite ripetute
mates["n"].value_counts().sort_index()

**Split.**

- Per n = 1…5 teniamo i puzzle con popolarità ≥ 50 e rating affidabile, e prendiamo lo stesso numero per ogni n (altrimenti dominerebbero i matti in 1 e 2). Mettiamo da parte `TEST_PER_N` puzzle per n come test; il resto va diviso 90/10 in train e validation.
- Per n = 6…10 c'è solo il test: la GNN **non** li vede in training, quindi lì misuriamo la generalizzazione. Applichiamo i filtri di qualità solo se restano abbastanza puzzle, altrimenti per n ≥ 8 non ne resterebbe quasi nessuno.

La deduplica per partita è fatta su tutto il dataset insieme, quindi nessun puzzle di test viene dalla stessa partita di un puzzle di training.

In [ ]:
good = (mates["Popularity"] >= 50) & (mates["RatingDeviation"] <= 90)
short = mates[good & (mates["n"] <= 5)].sample(frac=1, random_state=SEED).groupby("n").head(PER_N)
test_short = short.groupby("n").head(TEST_PER_N)
rest = short.drop(test_short.index)
val = rest.groupby("n").sample(frac=0.1, random_state=SEED)
train = rest.drop(val.index)

long_parts = []
for n, g in mates[mates["n"] >= 6].groupby("n"):
    if (good[g.index]).sum() >= LONG_PER_N:
        g = g[good[g.index]]
    long_parts.append(g.sample(min(len(g), LONG_PER_N), random_state=SEED))
test_long = pd.concat(long_parts)

pd.DataFrame({"train": train["n"].value_counts(), "val": val["n"].value_counts(),
              "test": pd.concat([test_short, test_long])["n"].value_counts()}).sort_index().fillna(0).astype(int)

## 2. Dagli scacchi ai grafi di TimeGNN

TimeGNN lavora su *event log* (case, evento, tempo) e predice l'evento successivo. La mappatura:

| TimeGNN | Scacchi |
|---|---|
| case | un puzzle |
| evento | una mossa (UCI); le classi sono 1968 mosse possibili + `EOS` |
| tempo | istante della mossa (simulato: i puzzle non hanno orologio) |
| feature del nodo | scacchiera **dopo** la mossa (64 case), scacco, matto, mosse rimaste, tempo |
| feature di sequenza | rating e n |

Ogni puzzle diventa un grafo a catena: mossa 1 → mossa 2 → … La label del nodo i è la mossa i+1 (`EOS` dopo il matto), quindi il nodo contiene esattamente la posizione in cui va trovata la mossa successiva.

**Scelte:**
- **Forma canonica.** Se a dare matto è il Nero, specchiamo scacchiera e mosse: il risolutore gioca sempre col Bianco e il modello impara ogni schema una volta sola.
- **Tempo simulato dal rating.** Tempo medio di 4·e^((rating−1500)/1000) secondi, più rumore log-normale; poi scalato con log1p(t)/log1p(600) in [0, 1].
- **Vocabolario fisso.** Un caso finto con tutte le mosse viene aggiunto solo al fit dell'encoder, così nessuna mossa di test risulta sconosciuta.

In [ ]:
SQUARES = [f"sq{i}" for i in range(64)]
T0 = pd.Timestamp("2020-01-01", tz="UTC")

# tutte le mosse UCI geometricamente possibili (1968)
MOVES = set()
for a in range(64):
    for b in range(64):
        dx, dy = abs(a % 8 - b % 8), abs(a // 8 - b // 8)
        if a != b and (dx == 0 or dy == 0 or dx == dy or dx * dy == 2):
            MOVES.add(chess.square_name(a) + chess.square_name(b))
for f in range(8):
    for d in (-1, 0, 1):
        if 0 <= f + d < 8:
            for p in "qrbn":
                MOVES.add(chess.square_name(48 + f) + chess.square_name(56 + f + d) + p)
                MOVES.add(chess.square_name(8 + f) + chess.square_name(f + d) + p)
MOVES = sorted(MOVES)
CLASS_ID = {m: i for i, m in enumerate(sorted(MOVES + ["EOS"]))}   # stesso ordine del LabelEncoder


def mirror(uci):
    m = chess.Move.from_uci(uci)
    return chess.Move(chess.square_mirror(m.from_square), chess.square_mirror(m.to_square), m.promotion).uci()


def canonical(fen, moves):
    # nel CSV al tratto c'è l'avversario: se è il Bianco, il risolutore è il Nero -> specchiamo
    board = chess.Board(fen)
    flip = board.turn == chess.WHITE
    if flip:
        board, moves = board.mirror(), [mirror(m) for m in moves]
    return board.fen(), moves, flip


def think_time(rating, rng=None):
    mean = 4 * np.exp((rating - 1500) / 1000)
    if rng is None:
        return mean                                   # in valutazione: il tempo medio
    return float(np.clip(rng.lognormal(np.log(mean), 0.6), 0.5, 600))


def puzzle_rows(pid, fen, moves, times, rating, n):
    # un puzzle -> righe dell'event log (una riga = un nodo = una mossa)
    board = chess.Board(fen)
    rows, t, solver_moves = [], 0.0, 0
    for i, m in enumerate(moves):
        board.push_uci(m)
        solver_moves += i % 2                         # mosse del risolutore: 1, 3, 5, ...
        t += times[i]
        row = {"case": pid, "ply": i, "move": m, "ts": T0 + pd.Timedelta(seconds=t),
               "t": min(np.log1p(times[i]) / np.log1p(600), 1.0),
               "check": str(int(board.is_check())), "mate": str(int(board.is_checkmate())),
               "left": n - solver_moves, "rating": rating, "n": n}
        for s in range(64):
            p = board.piece_at(s)
            row[SQUARES[s]] = p.symbol() if p else "."
        rows.append(row)
    return rows


def to_event_log(df, rng):
    rows = []
    for p in df.itertuples():
        fen, moves, _ = canonical(p.FEN, p.Moves.split())
        rows += puzzle_rows(p.PuzzleId, fen, moves, [think_time(p.Rating, rng) for _ in moves], p.Rating, p.n)
    return pd.DataFrame(rows)


rng = np.random.default_rng(SEED)
train_df, val_df = to_event_log(train, rng), to_event_log(val, rng)
T_MEAN = train_df["t"].mean()      # usato da M4c (tempo costante)
print(len(train_df), "nodi di training,", len(val_df), "di validation")
train_df[["case", "ply", "move", "check", "mate", "left", "t", "rating", "n"]].head(4)

`make_graphs` usa `EventLogTransformer` della libreria e poi applica due correzioni **ai dati**:

1. **Tempo causale.** Sull'arco i → i+1 mettiamo il tempo speso sulla mossa i+1. La libreria userebbe il tempo che manca alla fine del puzzle, che durante la risoluzione non si conosce. Nelle varianti senza tempo mettiamo 0.
2. **Self-loop nel modello time-decay.** `TimeAwareGATConv` non li aggiunge, quindi un nodo non vedrebbe le proprie feature: su un matto in 1 l'output sarebbe sempre lo stesso. Hanno tempo 0, quindi decadimento 1. `GATConv` del modello basic li aggiunge già da solo.

In [ ]:
def make_transformer(mode, use_time, train_df):
    tr = EventLogTransformer(case_col="case", event_col="move", time_col="ts", mode=mode,
                             cat_event=SQUARES + ["check", "mate"],
                             num_event=["left", "t"] if use_time else ["left"],
                             seq_cols=["rating", "n"], num_seq=["rating", "n"])
    vocab = pd.DataFrame({"case": "zzz_vocab", "ply": range(len(MOVES)), "move": MOVES, "ts": T0,
                          "check": "0", "mate": "0", "left": -1, "t": -1.0, "rating": -1.0, "n": -1})
    for c in SQUARES:
        vocab[c] = "."
    tr.fit(pd.concat([train_df, vocab], ignore_index=True))   # -1 = "mancante" per la libreria
    assert list(tr._event_encoder.classes_) == list(CLASS_ID)
    return tr


def make_graphs(tr, df, mode, use_time):
    df = df.sort_values(["case", "ply"]).reset_index(drop=True)
    cases = df["case"].unique()
    graphs = []
    for part in np.array_split(cases, max(1, len(cases) // 5000)):     # a blocchi per la RAM
        sub = df[df["case"].isin(part)].reset_index(drop=True)
        out = tr.transform(sub)
        times = sub.groupby("case")["t"].apply(list)                     # stesso ordine della libreria
        for g, y, t in zip(out.event_features, out.labels, times):
            n = g.num_nodes
            edges = g.edge_index.view(2, -1)
            et = torch.tensor(t[1:], dtype=torch.float) if use_time else torch.zeros(n - 1)
            if mode == "gat_basic":
                g.edge_index, g.edge_attr = edges, et.view(-1, 1)
            else:
                g.edge_index = torch.cat([edges, torch.arange(n).repeat(2, 1)], dim=1)
                g.time = torch.cat([et, torch.zeros(n)])
            graphs.append((g, y))
    return graphs


# controllo su un puzzle: label = mossa successiva, tempo sugli archi, self-loop
ids = {i: m for m, i in CLASS_ID.items()}
small = train_df[train_df["case"].isin(train_df["case"].unique()[:200])]
one = small[small["case"] == small["case"].iloc[0]]
for mode in ["gat_basic", "gat_time_decay"]:
    g, y = make_graphs(make_transformer(mode, True, small), one, mode, True)[0]
    print(mode, "| mosse:", one["move"].tolist(), "| label:", [ids[i] for i in y.view(-1).tolist()])
    print("   archi:", g.edge_index.tolist(), "| tempo:",
          (g.edge_attr.view(-1) if mode == "gat_basic" else g.time).numpy().round(2).tolist())

## 3. Valutazione: far risolvere i puzzle

Ogni sistema (GNN, baseline, LLM) viene valutato con la stessa funzione `solve`:
1. il sistema sceglie la mossa del risolutore (`choose`);
2. se ha seguito la soluzione, il difensore risponde con la mossa della soluzione; se ha deviato, risponde **Stockfish**;
3. il puzzle è **risolto** se arriva il matto entro n mosse. Sono accettati anche matti alternativi.

Il test set (corto + lungo) è lo stesso per tutti i sistemi, quindi i confronti sono appaiati.

In [ ]:
rows = []
for p in pd.concat([test_short, test_long]).itertuples():
    fen, moves, flip = canonical(p.FEN, p.Moves.split())
    rows.append({"case": p.PuzzleId, "fen": fen, "moves": " ".join(moves), "n": p.n, "rating": p.Rating,
                 "flip": flip, "fen_orig": p.FEN, "moves_orig": p.Moves})
test = pd.DataFrame(rows)
test.to_parquet("data/test.parquet")


def solve(p, choose, engine):
    sol = p["moves"].split()
    board = chess.Board(p["fen"])
    board.push_uci(sol[0])                        # la mossa dell'avversario è data
    played, on_line, first_ok = [sol[0]], True, None
    for k in range(p["n"]):
        m = choose(p, board, played)
        if m is None:                             # mossa illegale (solo LLM)
            return "illegal", bool(first_ok), played
        first_ok = (m == sol[1]) if first_ok is None else first_ok
        on_line = on_line and m == sol[2 * k + 1]
        board.push_uci(m)
        played.append(m)
        if board.is_checkmate():
            return "solved", first_ok, played
        if k == p["n"] - 1 or board.is_game_over():
            break
        reply = sol[2 * k + 2] if on_line else engine.play(board, chess.engine.Limit(time=0.05)).move.uci()
        board.push_uci(reply)
        played.append(reply)
        if board.is_game_over():
            break
    return "failed", first_ok, played


def evaluate(system, seed, choose, puzzles):
    engine = chess.engine.SimpleEngine.popen_uci(STOCKFISH)
    res = []
    for p in puzzles.to_dict("records"):
        t0 = time.time()
        status, first_ok, played = solve(p, choose, engine)
        res.append({"system": system, "seed": seed, "case": p["case"], "n": p["n"], "status": status,
                    "solved": status == "solved", "first_ok": first_ok,
                    "moves": " ".join(played), "sec": time.time() - t0})
    engine.quit()
    return pd.DataFrame(res)


print(len(test), "puzzle di test:", test["n"].value_counts().sort_index().to_dict())

## 4. Training delle varianti GNN

| Variante | Modello TimeGNN | Tempo |
|---|---|---|
| M1 | `DualGATModel` (basic) | no |
| M2 | `DualGATModel` | sì: feature `t` e tempo sugli archi (dentro l'attenzione) |
| M3 | `DualGATTimeAwareModel` (time-decay) | no (decadimento sempre 1) |
| M4 | `DualGATTimeAwareModel` | sì: l'attenzione verso il predecessore è moltiplicata per e^(−λt) |
| M4c | come M4 ma con **tempo costante** (media del training) | controllo |

M4c serve a capire se un eventuale vantaggio di M4 viene dall'*informazione* del tempo o solo dal fatto che il decadimento pesa il predecessore meno del nodo stesso.

Per ogni variante e seed: training con `train_epoch`/`evaluate_epoch` della libreria, early stopping sulla validation, pesi migliori. Poi valutazione su tutto il test. Una run già fatta (file presente in `OUT`) viene saltata, quindi la cella si può interrompere e riprendere.

Le mosse si scelgono **solo tra quelle legali**: la mossa con il punteggio più alto tra le legali.

In [ ]:
VARIANTS = {"M1": ("gat_basic", False), "M2": ("gat_basic", True), "M3": ("gat_time_decay", False),
            "M4": ("gat_time_decay", True), "M4c": ("gat_time_decay", True)}


def build_model(mode, n_features):
    params = dict(num_event_features=n_features, num_embedding_features=len(CLASS_ID),
                  output_dim=len(CLASS_ID), embedding_dims=64, gat_hidden_dim_event=64,
                  gat_hidden_dim_embed=128, gat_hidden_dim_concat=128, num_heads=4,
                  num_layers=2, dropout=0.1, use_batch_norm=True)
    if mode == "gat_basic":
        return DualGATModel(**params)
    return DualGATTimeAwareModel(lambda_decay=LAMBDA, **params)


def loader(graphs, shuffle):
    ds = CustomDataset([g for g, _ in graphs], [y for _, y in graphs])
    return DataLoader(ds, batch_size=BATCH, shuffle=shuffle, collate_fn=custom_collate_fn)


for v, (mode, use_time) in VARIANTS.items():
    todo = [s for s in SEEDS if not os.path.exists(f"{OUT}/{v}_s{s}.csv")]
    if not todo:
        print(v, "già fatto")
        continue
    tr_df, va_df = train_df.copy(), val_df.copy()
    if v == "M4c":
        tr_df["t"], va_df["t"] = T_MEAN, T_MEAN
    tr = make_transformer(mode, use_time, tr_df)
    train_g, val_g = make_graphs(tr, tr_df, mode, use_time), make_graphs(tr, va_df, mode, use_time)

    for seed in todo:
        torch.manual_seed(seed); np.random.seed(seed)
        model = build_model(mode, train_g[0][0].x.shape[1]).to(DEVICE)
        opt = torch.optim.Adam(model.parameters(), lr=LR)
        loss_fn = torch.nn.CrossEntropyLoss(ignore_index=-1)
        stopper, history, best = EarlyStopping(patience=PATIENCE), [], None
        train_loader, val_loader = loader(train_g, True), loader(val_g, False)
        for ep in range(1, EPOCHS + 1):
            t0 = time.time()
            tl, ta = train_epoch(model, train_loader, opt, loss_fn, DEVICE)
            vl, va = evaluate_epoch(model, val_loader, loss_fn, DEVICE)
            history.append({"system": v, "seed": seed, "epoch": ep, "train_loss": tl, "train_acc": ta,
                            "val_loss": vl, "val_acc": va})
            stop = stopper(vl)
            if stopper.best_loss_updated:
                best = copy.deepcopy(model.state_dict())
            print(f"{v} s{seed} epoca {ep}: train {tl:.3f}/{ta:.3f} | val {vl:.3f}/{va:.3f} | {time.time()-t0:.0f}s")
            if stop and ep >= MIN_EPOCHS:
                break
        model.load_state_dict(best)
        model.eval()
        torch.save(best, f"{OUT}/{v}_s{seed}.pt")
        pd.DataFrame(history).to_csv(f"{OUT}/{v}_s{seed}_history.csv", index=False)

        def choose(p, board, played):
            times = [think_time(p["rating"])] * len(played)
            df = pd.DataFrame(puzzle_rows("x", p["fen"], played, times, p["rating"], p["n"]))
            if v == "M4c":
                df["t"] = T_MEAN
            g, _ = make_graphs(tr, df, mode, use_time)[0]
            with torch.no_grad():
                logits = model(Batch.from_data_list([g]).to(DEVICE))[-1].cpu()
            legal = [m.uci() for m in board.legal_moves]
            return legal[int(logits[[CLASS_ID[m] for m in legal]].argmax())]

        res = evaluate(v, seed, choose, test)
        res.to_csv(f"{OUT}/{v}_s{seed}.csv", index=False)
        print(f"   risolti per n:", res.groupby("n")["solved"].mean().round(2).to_dict())
    del train_g, val_g

## 5. Baseline

- **random**: una mossa legale a caso.
- **greedy**: se c'è una mossa che dà matto la gioca, altrimenti preferisce uno scacco, poi una cattura. Risolve per costruzione tutti i matti in 1: ricorda che la GNN riconosce schemi ma non fa ricerca.

In [ ]:
random.seed(0)

def random_move(p, board, played):
    return random.choice([m.uci() for m in board.legal_moves])

def greedy_move(p, board, played):
    mates, checks, caps = [], [], []
    for m in board.legal_moves:
        if board.is_capture(m):
            caps.append(m.uci())
        if board.gives_check(m):
            b = board.copy(); b.push(m)
            (mates if b.is_checkmate() else checks).append(m.uci())
    return random.choice(mates or checks or caps or [m.uci() for m in board.legal_moves])

for name, fn in [("random", random_move), ("greedy", greedy_move)]:
    if not os.path.exists(f"{OUT}/{name}_s0.csv"):
        evaluate(name, 0, fn, test).to_csv(f"{OUT}/{name}_s0.csv", index=False)

## 6. LLM via Ollama

L'LLM viene valutato con la stessa `solve`, in **zero-shot**. Il prompt è quello della traccia ("find mate in n, explain step by step"), con tre aggiunte:
- la scacchiera disegnata, perché gli LLM leggono male il FEN da solo;
- la lista delle mosse legali, perché anche la GNN sceglie solo tra mosse legali;
- la richiesta di rispondere in JSON (`reasoning` + `move`), con il formato imposto da Ollama.

Altre scelte:
- L'LLM vede la posizione **reale**, non quella specchiata.
- **Temperatura 0**, quindi basta una run.
- **Ragionamento interno spento.** Acceso, qwen3 genera migliaia di token (oltre 80 s per mossa) e spesso non arriva a rispondere.
- `num_predict` limita la risposta, per evitare che il modello vada in loop.

Si usano i primi `LLM_PER_N` puzzle per n = 1…5 più tutti i puzzle lunghi. Le righe vengono salvate una alla volta, quindi si può interrompere e riprendere. Stima: circa 7 s a puzzle per n = 1…5, circa 18 s per i lunghi.

In [ ]:
SCHEMA = {"type": "object", "properties": {"reasoning": {"type": "string"}, "move": {"type": "string"}},
          "required": ["reasoning", "move"]}


def make_prompt(board, moves_left):
    side = "White" if board.turn == chess.WHITE else "Black"
    rows = str(board).split("\n")
    drawing = "\n".join(f"{8 - i} {r}" for i, r in enumerate(rows)) + "\n  a b c d e f g h"
    return (f"You are a chess expert. Solve this chess puzzle.\n\nFEN: {board.fen()}\n\n"
            f"Board (uppercase = White, lowercase = Black):\n{drawing}\n\n"
            f"{side} to move. {side} can deliver checkmate in {moves_left} move(s) or fewer.\n"
            f"Legal moves (UCI): {' '.join(m.uci() for m in board.legal_moves)}\n\n"
            "Think step by step, then choose the best move. Answer in JSON with two fields: "
            "\"reasoning\" (a short step-by-step explanation, at most 5 sentences) and "
            "\"move\" (the move in UCI format, e.g. e2e4).")


def ask_llm(prompt):
    body = {"model": LLM_MODEL, "stream": False, "format": SCHEMA,
            "messages": [{"role": "user", "content": prompt}],
            "options": {"temperature": 0, "seed": 0, "num_ctx": 8192, "num_predict": 400}}
    if LLM_THINK is not None:
        body["think"] = LLM_THINK
    for attempt in range(3):                     # Ollama ogni tanto risponde con un errore: riproviamo
        r = requests.post("http://localhost:11434/api/chat", json=body, timeout=900).json()
        if "message" in r:
            break
        time.sleep(5)
    else:
        raise RuntimeError(f"Ollama: {r}")       # rilanciando la cella si riparte da qui
    try:
        ans = json.loads(r["message"]["content"])
    except json.JSONDecodeError:                 # risposta troncata
        ans = {"reasoning": r["message"]["content"], "move": ""}
    return str(ans.get("move", "")), str(ans.get("reasoning", "")), r.get("eval_count", 0)


def parse_move(board, text):
    text = re.sub(r"^\d+\.+\s*", "", text.strip())       # toglie "1." o "1..."
    for parse in (lambda s: chess.Move.from_uci(s.lower()), board.parse_san):
        try:
            m = parse(text)
            if m in board.legal_moves:
                return m.uci()
        except ValueError:
            pass
    return None


log = []
def llm_choose(p, board, played):
    real = board.mirror() if p["flip"] else board        # la posizione vera, non specchiata
    answer, reasoning, tokens = ask_llm(make_prompt(real, p["n"] - (len(played) - 1) // 2))
    m = parse_move(real, answer)
    log.append({"answer": answer, "reasoning": reasoning, "tokens": tokens})
    if m is None:
        return None
    return mirror(m) if p["flip"] else m


name = "LLM_" + LLM_MODEL.replace(":", "_")
path = f"{OUT}/{name}_s0.csv"
llm_test = pd.concat([test[test["n"] <= 5].groupby("n").head(LLM_PER_N), test[test["n"] >= 6]])
done = set(pd.read_csv(path)["case"]) if os.path.exists(path) else set()
todo = llm_test[~llm_test["case"].isin(done)]
print(f"{len(done)} già fatti, {len(todo)} da fare")
for i in range(len(todo)):
    log = []
    row = evaluate(name, 0, llm_choose, todo.iloc[[i]])
    row["calls"] = len(log)
    row["tokens"] = sum(l["tokens"] for l in log)
    row["answers"] = json.dumps([l["answer"] for l in log])
    row["reasoning"] = log[0]["reasoning"] if log else ""
    row.to_csv(path, mode="a", header=not os.path.exists(path), index=False)
    if (i + 1) % 20 == 0:
        print(f"  {i + 1}/{len(todo)}")

## 7. Risultati

**Metriche:**
- `solved`: matto entro n mosse;
- `first_ok`: prima mossa uguale alla soluzione;
- `sec`: secondi per puzzle.

Per la GNN si fa la media dei 3 seed. n = 9 e 10 sono pochissimi puzzle, quindi li uniamo nel gruppo "9-10".

In [ ]:
res = pd.concat([pd.read_csv(f) for f in glob.glob(f"{OUT}/*_s*.csv") if "history" not in f], ignore_index=True)
res["gruppo"] = res["n"].apply(lambda n: str(n) if n <= 8 else "9-10")
ORDER = [g for g in ["1", "2", "3", "4", "5", "6", "7", "8", "9-10"] if g in set(res["gruppo"])]
LLM = name
SYSTEMS = ["M1", "M2", "M3", "M4", "M4c", LLM, "greedy", "random"]

def by_n(df, col="solved"):
    return (df.groupby(["system", "seed", "gruppo"])[col].mean().groupby(["system", "gruppo"]).mean()
              .unstack()[ORDER].reindex(SYSTEMS).mul(100).round(1))

print("Puzzle risolti (%), test completo (LLM sul suo sottoinsieme):")
display(by_n(res))
print("Prima mossa corretta (%):")
display(by_n(res, "first_ok"))
print("Secondi per puzzle:")
display(res.groupby("system")["sec"].mean().reindex(SYSTEMS).round(2))

In [ ]:
# stessi puzzle per tutti: il sottoinsieme valutato dall'LLM
common = res[res["case"].isin(set(res.loc[res.system == LLM, "case"]))]
table = by_n(common)
fig, ax = plt.subplots(figsize=(9, 4))
for s in ["M1", "M4", LLM, "greedy", "random"]:
    ax.plot(ORDER, table.loc[s], marker="o", label=s)
if "6" in ORDER:
    ax.axvline(ORDER.index("6") - 0.5, color="gray", linestyle="--")
    ax.text(ORDER.index("6") - 0.4, 90, "fuori dal training della GNN", fontsize=8, color="gray")
ax.set_xlabel("n (matto in n)"); ax.set_ylabel("% puzzle risolti"); ax.set_ylim(0, 100)
ax.set_title("Tasso di soluzione per n"); ax.legend(fontsize=8)
plt.show()

hist = pd.concat([pd.read_csv(f) for f in glob.glob(f"{OUT}/*_history.csv")])
for (s, seed), h in hist.groupby(["system", "seed"]):
    plt.plot(h["epoch"], h["val_loss"], label=f"{s} s{seed}")
plt.xlabel("epoca"); plt.ylabel("validation loss"); plt.legend(fontsize=6, ncol=3); plt.show()

### Domanda (1): GNN vs LLM per n

Test di McNemar sugli stessi puzzle. Per la GNN (M4) un puzzle conta come risolto se lo risolvono almeno 2 seed su 3. Confrontiamo anche la GNN con il greedy. Il vincitore viene indicato solo se p < 0,05.

In [ ]:
def mcnemar(a, b):
    only_a, only_b = int((a & ~b).sum()), int((~a & b).sum())
    p = binomtest(only_a, only_a + only_b).pvalue if only_a + only_b else 1.0
    return only_a, only_b, p

gnn = res[res.system == "M4"].groupby("case")["solved"].mean() >= 2 / 3
rows = []
for other in [LLM, "greedy"]:
    X = res[res.system == other].set_index("case")
    for grp in ORDER:
        c = X.index[X["gruppo"] == grp]
        a, b, p = mcnemar(gnn.loc[c], X.loc[c, "solved"])
        rows.append({"contro": other, "n": grp, "puzzle": len(c), "GNN %": round(100 * gnn.loc[c].mean(), 1),
                     "altro %": round(100 * X.loc[c, "solved"].mean(), 1), "solo GNN": a, "solo altro": b,
                     "p": f"{p:.2g}", "vince": "pari" if p >= 0.05 else ("GNN" if a > b else other)})
pd.DataFrame(rows)

### Domanda (2): il tempo aiuta?

Confronti a coppie sugli stessi puzzle: McNemar per ogni seed e Wilcoxon sulla media dei seed per puzzle.
- **M1 → M2**: effetto del tempo nel modello basic.
- **M3 → M4**: effetto del tempo nel modello time-decay.
- **M4c → M4**: effetto dell'*informazione* temporale, a parità di decadimento.
- **M3 → M4c**: effetto della sola ripesatura introdotta dal decadimento.

In [ ]:
rows = []
for part, df in [("n 1-5", res[res.n <= 5]), ("n 6-10", res[res.n >= 6])]:
    for a, b in [("M1", "M2"), ("M3", "M4"), ("M4c", "M4"), ("M3", "M4c")]:
        for seed in SEEDS:
            A = df[(df.system == a) & (df.seed == seed)].set_index("case")["solved"]
            B = df[(df.system == b) & (df.seed == seed)].set_index("case")["solved"].loc[A.index]
            x, y, p = mcnemar(A, B)
            rows.append({"test": part, "confronto": f"{a} → {b}", "seed": seed,
                         "diff (punti)": round(100 * (B.mean() - A.mean()), 1), "p": round(p, 3)})
        A = df[df.system == a].groupby("case")["solved"].mean()
        B = df[df.system == b].groupby("case")["solved"].mean().loc[A.index]
        p = wilcoxon(B, A).pvalue if (B != A).any() else 1.0
        rows.append({"test": part, "confronto": f"{a} → {b}", "seed": "media",
                     "diff (punti)": round(100 * (B.mean() - A.mean()), 1), "p": round(p, 3)})
pd.DataFrame(rows)

## 8. Valutazione umana (blind)

Foglio con 20 puzzle: per ognuno due guide anonime, A e B, in ordine casuale. Una viene dalla GNN (M4, seed 0), l'altra dall'LLM (mossa e spiegazione). I valutatori danno un voto da 1 a 5 per **correttezza** e **chiarezza**; la chiave è in un file separato.

Limite: la GNN non produce spiegazioni, quindi il blind è solo parziale.

In [ ]:
random.seed(0)
G = res[(res.system == "M4") & (res.seed == 0)].set_index("case")
L = res[res.system == LLM].set_index("case")
sample = test[test["case"].isin(L.index)].sample(min(20, len(L)), random_state=0)

def first_move_san(p, played):
    moves = str(played).split()
    if len(moves) < 2:
        return "(nessuna mossa valida)"
    b = chess.Board(p["fen_orig"]); b.push_uci(p["moves_orig"].split()[0])
    m = chess.Move.from_uci(mirror(moves[1]) if p["flip"] else moves[1])
    return b.san(m)

sheet, key = [], []
for p in sample.to_dict("records"):
    g = f"Mossa consigliata: {first_move_san(p, G.loc[p['case'], 'moves'])}"
    l = f"Mossa consigliata: {first_move_san(p, L.loc[p['case'], 'moves'])}. {L.loc[p['case'], 'reasoning']}"
    swap = random.random() < 0.5
    b = chess.Board(p["fen_orig"]); b.push_uci(p["moves_orig"].split()[0])
    sheet.append({"puzzle": p["case"], "FEN": b.fen(), "n": p["n"], "guida_A": l if swap else g,
                  "guida_B": g if swap else l, "A_correttezza": "", "A_chiarezza": "",
                  "B_correttezza": "", "B_chiarezza": ""})
    key.append({"puzzle": p["case"], "A": LLM if swap else "M4", "B": "M4" if swap else LLM})
pd.DataFrame(sheet).to_csv(f"{OUT}/valutazione_umana.csv", index=False)
pd.DataFrame(key).to_csv(f"{OUT}/valutazione_umana_CHIAVE.csv", index=False)
pd.DataFrame(sheet).head(3)

## Note per il report

- **Tempo simulato.** Il tempo è derivato dal rating, che il modello conosce già, quindi non porta informazione nuova. La verifica vera della domanda (2) richiederebbe tempi reali, cioè partite con l'orologio.
- **n ≥ 6.** La GNN non li ha visti in training (le feature `n` e `left` escono dal range 1–5): è un test di generalizzazione.
- **Lunghezza e difficoltà.** Il rating cresce con n, quindi lunghezza e difficoltà non sono separabili.
- **LLM.** Modello locale da 8B, zero-shot, senza ragionamento interno: la conclusione vale per questa classe di modelli.
- **Libreria.** TimeGNN non è stata modificata. Usiamo le sue classi (`EventLogTransformer`, `DualGATModel`, `DualGATTimeAwareModel`, `train_epoch`, `EarlyStopping`); le correzioni (self-loop, tempo causale) sono applicate ai dati, più una riga di compatibilità con PyG.